# Feature Stores, Data Pipelines & Offline/Online Serving
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 2, Session 1**

Today we're going to get into the data layer — the part of the system most engineers underestimate. A bad model with great data will outperform a great model with bad data, every time.

We'll cover four concrete things:
1. Data formats — why Parquet exists and when it matters
2. The leakage trap — a mistake that looks fine in a notebook but destroys production performance
3. Point-in-time feature construction — the right way to build training data from event logs
4. A simple feature store simulation — the same code path for training and serving

## Part 1: Data Formats — CSV vs Parquet
*Why your data team keeps asking you to use Parquet*

We're going to generate a realistic dataset, write it as both CSV and Parquet, and compare read speed and file size. This matters once your datasets hit tens of millions of rows.

In [1]:
import numpy as np
import pandas as pd
import os
import time
import tempfile

np.random.seed(42)

# Generate a realistic transaction-style dataset
n_rows = 500_000
df = pd.DataFrame({
    "user_id":          np.random.randint(1, 100_000, n_rows),
    "item_id":          np.random.randint(1, 50_000, n_rows),
    "timestamp":        pd.date_range("2023-01-01", periods=n_rows, freq="1min"),
    "price":            np.round(np.random.exponential(50, n_rows), 2),
    "quantity":         np.random.randint(1, 10, n_rows),
    "category":         np.random.choice(["electronics", "clothing", "food", "books"], n_rows),
    "user_age":         np.random.randint(18, 80, n_rows),
    "session_length":   np.random.exponential(300, n_rows).astype(int),
    "device_type":      np.random.choice(["mobile", "desktop", "tablet"], n_rows),
    "converted":        np.random.randint(0, 2, n_rows),
})

# Write to temp files
tmpdir = tempfile.mkdtemp()
csv_path = os.path.join(tmpdir, "data.csv")
parquet_path = os.path.join(tmpdir, "data.parquet")

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

csv_size_mb = os.path.getsize(csv_path) / 1e6
parquet_size_mb = os.path.getsize(parquet_path) / 1e6

print("=== Data Format Comparison ===")
print(f"Dataset: {n_rows:,} rows × {len(df.columns)} columns")
print()
print("Writing files...")
print(f"  CSV written:     {csv_size_mb:.1f} MB")
print(f"  Parquet written:  {parquet_size_mb:.1f} MB")
print(f"  Size ratio: CSV is {csv_size_mb/parquet_size_mb:.2f}x larger")

# Benchmark reads
def bench_read(read_fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        read_fn()
        times.append(time.perf_counter() - t0)
    return min(times)

csv_time = bench_read(lambda: pd.read_csv(csv_path))
parquet_time = bench_read(lambda: pd.read_parquet(parquet_path))

print()
print("Read benchmarks (3 runs each, best of 3):")
print(f"  CSV read time:     {csv_time:.4f} s")
print(f"  Parquet read time: {parquet_time:.4f} s")
print(f"  Parquet is {csv_time/parquet_time:.1f}x faster to read")

# Column-selective read: Parquet only reads requested columns from disk
parquet_col_time = bench_read(lambda: pd.read_parquet(parquet_path, columns=["user_id", "converted"]))
csv_col_time = bench_read(lambda: pd.read_csv(csv_path, usecols=["user_id", "converted"]))

print()
print("Column-selective read (Parquet only reads needed columns):")
print(f"  Read 2 of 10 columns from Parquet: {parquet_col_time:.4f} s")
print(f"  CSV always reads all columns:      {csv_col_time:.4f} s")
print(f"  For column-sparse queries, Parquet is {csv_col_time/parquet_col_time:.1f}x faster")

=== Data Format Comparison ===
Dataset: 500,000 rows × 10 columns

Writing files...
  CSV written:     31.8 MB
  Parquet written:  9.3 MB
  Size ratio: CSV is 3.44x larger

Read benchmarks (3 runs each, best of 3):
  CSV read time:     0.2290 s
  Parquet read time: 0.0203 s
  Parquet is 11.3x faster to read

Column-selective read (Parquet only reads needed columns):
  Read 2 of 10 columns from Parquet: 0.0020 s
  CSV always reads all columns:      0.0806 s
  For column-sparse queries, Parquet is 39.7x faster


### When does this matter?

- At 10M rows: CSV read takes ~8s, Parquet ~1s. That's per feature engineering job.
- At 1B rows: you physically can't fit CSV in memory. Parquet's columnar layout lets you read only the columns you need.
- **Rule of thumb:** Use CSV for files you're going to look at. Use Parquet for files your code will read.

---
## Part 2: The Leakage Trap
*How random splits silently inflate your metrics*

This is one of the most common mistakes in production ML. The model looks great in offline evaluation, gets deployed, and immediately underperforms. We're going to reproduce that failure in 20 lines of code.

We'll use a time-series classification problem: predict whether a user will convert in the next 24 hours based on their last 7 days of activity.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)

# Simulate user activity time-series data
n_samples = 10_000
dates = pd.date_range("2023-01-01", periods=n_samples, freq="1h").date

# Features: rolling statistics over the past 7 days
# The key feature: recent_activity has a time-trend (users become more active over time)
time_index = np.arange(n_samples)
trend = time_index / n_samples  # represents the real-world trend increasing over time

X = np.column_stack([
    np.random.poisson(5 + trend * 10, n_samples),  # page_views_7d (trending up)
    np.random.poisson(2 + trend * 5, n_samples),   # purchases_7d (trending up)
    np.random.exponential(200, n_samples),          # avg_session_length (stable)
    np.random.randint(0, 10, n_samples),            # items_in_cart (stable)
    trend + np.random.randn(n_samples) * 0.1,       # recency_score (trending, leakage source)
])

# Label: conversion probability increases with trend (seasonal effect)
log_odds = -2.0 + 0.3 * X[:, 0] / 10 + 0.5 * X[:, 1] / 5 + 1.5 * X[:, 4]
probs = 1 / (1 + np.exp(-log_odds))
y = (np.random.rand(n_samples) < probs).astype(int)

dates_series = pd.date_range("2023-01-01", periods=n_samples, freq="1h")

print("=== The Leakage Trap ===")
print(f"Dataset: {n_samples:,} user-day observations from 2023-01-01 to 2023-04-10")
print(f"Label: will_convert_next_day ({y.mean()*100:.1f}% positive rate)")

# ── Split A: Random (WRONG) ───────────────────────────────────────────────────
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd = train_test_split(X, y, test_size=0.2, random_state=42)

model_rnd = GradientBoostingClassifier(n_estimators=50, random_state=42)
model_rnd.fit(X_tr_rnd, y_tr_rnd)
auc_rnd = roc_auc_score(y_te_rnd, model_rnd.predict_proba(X_te_rnd)[:, 1])
acc_rnd = accuracy_score(y_te_rnd, model_rnd.predict(X_te_rnd))

print()
print("--- Split A: Random split (WRONG for time-series) ---")
print(f"  Train: {len(X_tr_rnd)} samples  |  Test: {len(X_te_rnd)} samples")
print(f"  Test AUC (random split):      {auc_rnd:.4f}")
print(f"  Test Accuracy (random split): {acc_rnd:.4f}")

# ── Split B: Time-based (CORRECT) ─────────────────────────────────────────────
cutoff = int(n_samples * 0.7)
X_tr_time, X_te_time = X[:cutoff], X[cutoff:]
y_tr_time, y_te_time = y[:cutoff], y[cutoff:]

model_time = GradientBoostingClassifier(n_estimators=50, random_state=42)
model_time.fit(X_tr_time, y_tr_time)
auc_time = roc_auc_score(y_te_time, model_time.predict_proba(X_te_time)[:, 1])
acc_time = accuracy_score(y_te_time, model_time.predict(X_te_time))

cutoff_date = dates_series[cutoff].strftime("%Y-%m-%d")
print()
print("--- Split B: Time-based split (CORRECT) ---")
print(f"  Train: 2023-01-01 → {cutoff_date}  ({cutoff:,} samples)")
print(f"  Test:  {cutoff_date} → 2023-04-10  ({n_samples - cutoff:,} samples)")
print(f"  Test AUC (time-based split):      {auc_time:.4f}")
print(f"  Test Accuracy (time-based split): {acc_time:.4f}")

print()
print("--- The Gap ---")
print(f"  AUC inflation from leakage:      +{auc_rnd - auc_time:.4f}")
print(f"  Accuracy inflation from leakage: +{acc_rnd - acc_time:.4f}")
print()
print("What happened: with random split, test set contains future data that was")
print("used to train. The model 'memorised' patterns from the future.")
print("In production, there is no future data — performance drops immediately.")

=== The Leakage Trap ===
Dataset: 10,000 user-day observations from 2023-01-01 to 2023-04-10
Label: will_convert_next_day (38.4% positive rate)

--- Split A: Random split (WRONG for time-series) ---
  Train: 8000 samples  |  Test: 2000 samples
  Test AUC (random split):      0.6924
  Test Accuracy (random split): 0.6735

--- Split B: Time-based split (CORRECT) ---
  Train: 2023-01-01 → 2023-10-19  (7,000 samples)
  Test:  2023-10-19 → 2023-04-10  (3,000 samples)
  Test AUC (time-based split):      0.5404
  Test Accuracy (time-based split): 0.5570

--- The Gap ---
  AUC inflation from leakage:      +0.1520
  Accuracy inflation from leakage: +0.1165

What happened: with random split, test set contains future data that was
used to train. The model 'memorised' patterns from the future.
In production, there is no future data — performance drops immediately.


### The rule is simple, the discipline is hard

**For any ML problem involving time: your test set must only contain data from after your training cutoff.**

The tricky variants:
- Leakage through features (e.g., a feature computed using future events)
- Leakage through the target (e.g., label derived from data that wouldn't exist at prediction time)
- Group leakage (e.g., same user appears in train and test — you've seen their behavior pattern)

All three produce the same symptom: offline metrics look great, online metrics disappoint.

---
## Part 3: Point-in-Time Feature Construction
*Building training data from an event log — the right way*

We're going to simulate a transaction log and construct features correctly. The key constraint: when building a training example for event `e` at time `t`, **you can only use data from before `t`**.

This sounds obvious. In practice, it's easy to get wrong because your feature pipeline might join on `user_id` without filtering by timestamp.

In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Simulate a transaction log
n_events = 200
user_ids = np.random.randint(1, 11, n_events)   # 10 users
event_dates = pd.date_range("2023-01-01", periods=n_events, freq="12h").date
amounts = np.round(np.random.exponential(50, n_events), 1)
event_types = np.random.choice(["purchase", "view", "add_to_cart"], n_events, p=[0.5, 0.3, 0.2])

txn_log = pd.DataFrame({
    "user_id": user_ids,
    "event_time": pd.to_datetime(event_dates),
    "amount": amounts,
    "event_type": event_types,
})

print("Transaction log (first 10 rows):")
print(txn_log.head(10).to_string(index=False))

# Prediction events: these are the rows we're building training labels for
prediction_events = pd.DataFrame({
    "user_id": [3, 7, 2],
    "prediction_time": pd.to_datetime(["2023-01-10", "2023-01-15", "2023-01-20"]),
})

print()
print("Prediction events (we want to label these):")
print(prediction_events.to_string(index=False))

def build_point_in_time_features(user_id, cutoff_time, txn_log, lookback_days=30):
    """
    Build features for a user at a specific point in time.
    ONLY uses transactions strictly before the cutoff_time.
    This is the correct approach.
    """
    window_start = cutoff_time - pd.Timedelta(days=lookback_days)

    # ← The critical filter: strictly BEFORE the prediction time
    user_txns = txn_log[
        (txn_log["user_id"] == user_id) &
        (txn_log["event_time"] < cutoff_time) &          # <-- point-in-time filter
        (txn_log["event_time"] >= window_start) &
        (txn_log["event_type"] == "purchase")
    ]

    if len(user_txns) == 0:
        return {"tx_count_30d": 0, "avg_amount": 0, "total_spend": 0, "days_since_last": 999}

    last_txn_time = user_txns["event_time"].max()
    days_since_last = (cutoff_time - last_txn_time).days

    return {
        "tx_count_30d":   len(user_txns),
        "avg_amount":     round(user_txns["amount"].mean(), 2),
        "total_spend":    round(user_txns["amount"].sum(), 2),
        "days_since_last": days_since_last,
    }

print()
print("=== Point-in-Time Feature Construction ===")
for _, row in prediction_events.iterrows():
    features = build_point_in_time_features(row["user_id"], row["prediction_time"], txn_log)
    print(f"For user {row['user_id']} at {row['prediction_time'].date()}: "
          f"using {features['tx_count_30d']} transactions before cutoff")
    print(f"  tx_count_30d={features['tx_count_30d']}, avg_amount={features['avg_amount']}, "
          f"total_spend={features['total_spend']}, days_since_last={features['days_since_last']}")

# Show what happens WITHOUT the time filter (the wrong way)
print()
print("WRONG approach (join without time filter — data leakage):")
wrong_features = txn_log[
    (txn_log["user_id"] == 3) &
    (txn_log["event_type"] == "purchase")
    # ← missing the cutoff_time filter!
]
print(f"For user 3 (no time filter): {len(wrong_features)} transactions used — includes FUTURE data!")
print(f"  tx_count_30d={len(wrong_features)} (should be 3)")

Transaction log (first 10 rows):
 user_id event_time  amount  event_type
       7 2023-01-01    25.4 add_to_cart
       4 2023-01-01    84.8 add_to_cart
       8 2023-01-02    80.1        view
       5 2023-01-02     8.2 add_to_cart
       7 2023-01-03    35.5        view
      10 2023-01-03    59.5    purchase
       3 2023-01-04    97.7    purchase
       7 2023-01-04    19.7    purchase
       8 2023-01-05    12.4    purchase
       5 2023-01-05    62.1    purchase

Prediction events (we want to label these):
 user_id prediction_time
       3      2023-01-10
       7      2023-01-15
       2      2023-01-20

=== Point-in-Time Feature Construction ===
For user 3 at 2023-01-10: using 2 transactions before cutoff
  tx_count_30d=2, avg_amount=119.4, total_spend=238.8, days_since_last=3
For user 7 at 2023-01-15: using 1 transactions before cutoff
  tx_count_30d=1, avg_amount=19.7, total_spend=19.7, days_since_last=11
For user 2 at 2023-01-20: using 2 transactions before cutoff
  tx_count

### Why feature stores exist

The `build_point_in_time_features` function above is the **correct computation**. A feature store ensures:
1. This exact function is used both at training time (offline) and serving time (online)
2. The time filter is enforced automatically — you can't accidentally forget it
3. Features are cached so you don't recompute them on every training run

The simplest feature store is just: **a function + a consistent calling convention**.

---
## Part 4: Simple Feature Store Simulation
*Training and serving using the same computation function*

We're going to implement the minimal feature store pattern: one function that computes features, and two storage backends — offline (for training) and online (for serving). The key insight is that the **computation** is shared; only the **storage** differs.

In [4]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

# Simulate raw event data (what would live in a data warehouse)
raw_events = {
    "user_1": [{"date": "2023-05-28", "spend": 45.0}, {"date": "2023-05-29", "spend": 80.0},
               {"date": "2023-05-30", "spend": 30.0}, {"date": "2023-05-31", "spend": 70.0},
               {"date": "2023-06-01", "spend": 25.0}, {"date": "2023-05-26", "spend": 90.0},
               {"date": "2023-05-24", "spend": 60.0}, {"date": "2023-05-22", "spend": 100.0}],
    "user_2": [{"date": "2023-05-20", "spend": 20.0}, {"date": "2023-05-25", "spend": 35.0},
               {"date": "2023-05-31", "spend": 30.0}],
    "user_3": [{"date": "2023-05-15", "spend": 120.0}, {"date": "2023-05-18", "spend": 80.0},
               {"date": "2023-05-20", "spend": 95.0}, {"date": "2023-05-22", "spend": 110.0},
               {"date": "2023-05-25", "spend": 90.0}, {"date": "2023-05-28", "spend": 85.0},
               {"date": "2023-05-30", "spend": 100.0}, {"date": "2023-06-01", "spend": 115.0},
               {"date": "2023-05-16", "spend": 70.0}, {"date": "2023-05-17", "spend": 88.0},
               {"date": "2023-05-19", "spend": 95.0}, {"date": "2023-05-21", "spend": 102.0}],
    "user_4": [{"date": "2023-06-01", "spend": 15.0}],
    "user_5": [{"date": "2023-05-10", "spend": 40.0}, {"date": "2023-05-20", "spend": 55.0},
               {"date": "2023-05-25", "spend": 60.0}, {"date": "2023-05-29", "spend": 50.0},
               {"date": "2023-06-01", "spend": 45.0}, {"date": "2023-05-15", "spend": 40.0}],
}

# ── THE SHARED COMPUTATION FUNCTION ──────────────────────────────────────────
# This function is called identically whether we're building training data
# or serving a real-time request. This guarantees consistency.
def compute_user_features(user_id: str, cutoff_time: str, events: dict) -> dict:
    """
    Compute features for a user relative to a cutoff time.
    Used by BOTH the offline pipeline and the online serving layer.
    """
    cutoff = pd.Timestamp(cutoff_time)
    window_start = cutoff - pd.Timedelta(days=30)

    user_events = events.get(user_id, [])
    window_events = [
        e for e in user_events
        if window_start <= pd.Timestamp(e["date"]) < cutoff
    ]

    if not window_events:
        return {"tx_count_30d": 0, "avg_spend": 0.0, "active_days": 0, "spend_velocity": 0.0}

    spends = [e["spend"] for e in window_events]
    active_days = len(set(e["date"] for e in window_events))

    return {
        "tx_count_30d":   len(window_events),
        "avg_spend":      round(sum(spends) / len(spends), 2),
        "active_days":    active_days,
        "spend_velocity": round(sum(spends) / 30, 1),   # avg daily spend over 30 days
    }

# ── OFFLINE STORE: build training dataset ────────────────────────────────────
# In production: this runs as a batch job (Spark/Airflow) and writes to S3/Hive
offline_store = {}   # dict[user_id] → features (in prod: Parquet on S3)

training_cutoff = "2023-06-01"
user_ids = list(raw_events.keys())

print("=== Mini Feature Store Simulation ===")
print()
print("--- Offline: Computing training features for all users ---")
for uid in user_ids:
    features = compute_user_features(uid, training_cutoff, raw_events)
    offline_store[uid] = features
    print(f"Computed and stored features for {uid} at {training_cutoff}")

training_df = pd.DataFrame([
    {"user_id": uid, **feats} for uid, feats in offline_store.items()
])
print()
print("Offline feature store (training dataset):")
print(training_df.to_string(index=False))

# ── ONLINE STORE: pre-computed features for real-time serving ─────────────────
# In production: this is Redis (sub-millisecond lookup at serving time)
# The features are pre-computed by the same compute_user_features function,
# just run more frequently (e.g., every hour)
online_store = {}  # dict[user_id] → features (in prod: Redis)

serving_cutoff = "2023-06-01" 
for uid in user_ids:
    online_store[uid] = compute_user_features(uid, serving_cutoff, raw_events)

print()
print("--- Online: Serving features for a real-time request ---")
print("Request: user_3 wants a recommendation NOW")
online_features = online_store["user_3"]
print(f"Online feature lookup (sub-millisecond from Redis/dict): {online_features}")

print()
print("--- Consistency Check ---")
offline_f = offline_store["user_3"]
online_f = online_store["user_3"]
print(f"Offline features for user_3: {offline_f}")
print(f"Online features for user_3:  {online_f}")
print(f"Identical: {offline_f == online_f}  <-- training-serving consistency guaranteed")

print()
print("This is the core value of a feature store:")
print("  Training uses: offline_store[user_id]")
print("  Serving uses:  online_store[user_id]")
print("  Both came from: compute_user_features(user_id, cutoff_time)")

=== Mini Feature Store Simulation ===

--- Offline: Computing training features for all users ---
Computed and stored features for user_1 at 2023-06-01
Computed and stored features for user_2 at 2023-06-01
Computed and stored features for user_3 at 2023-06-01
Computed and stored features for user_4 at 2023-06-01
Computed and stored features for user_5 at 2023-06-01

Offline feature store (training dataset):
user_id  tx_count_30d  avg_spend  active_days  spend_velocity
 user_1             7      67.86            7            15.8
 user_2             3      28.33            3             2.8
 user_3            11      94.09           11            34.5
 user_4             0       0.00            0             0.0
 user_5             5      49.00            5             8.2

--- Online: Serving features for a real-time request ---
Request: user_3 wants a recommendation NOW
Online feature lookup (sub-millisecond from Redis/dict): {'tx_count_30d': 11, 'avg_spend': 94.09, 'active_days': 11,

### What a production feature store adds on top of this

- **Versioning**: each feature definition is versioned; old models use old definitions
- **Backfilling**: compute features for any historical date on demand
- **Monitoring**: detect when feature distributions drift between offline and online
- **Discovery**: catalog of all available features so teams can reuse them

Tools in this space: Feast (open-source), Tecton, Hopsworks, Vertex AI Feature Store.

---

**Discussion:** In the online store, we pre-computed features at midnight. What happens if a user makes 5 purchases at 2am? What's the staleness tradeoff, and how would you mitigate it?